# Getting Started with Occhio

This tutorial walks through the core occhio workflow: defining a sparse
feature distribution, training an autoencoder to compress it, and
inspecting the geometric structure of the learned representations.

Along the way you will see **superposition** in action -- the phenomenon
where a neural network packs more features than it has dimensions by
overlapping their representations.

Requires: `pip install occhio` (or editable install from this repo).

In [1]:
# -- Imports ------------------------------------------------------------------

import torch
from occhio import ToyModel
from occhio.autoencoders import TiedLinearRelu
from occhio.distributions import SparseUniform
from occhio.visualization import plot_embedding

# Reproducibility
torch.manual_seed(42)

## 1. Create a sparse data distribution

`SparseUniform` generates samples where each of `n_features` features is
independently active with probability `p_active`.  When active, its value
is drawn from Uniform(0, 1); otherwise it is zero.

With `p_active=0.05`, roughly 5% of features fire in any given sample.
This sparsity is what makes superposition possible -- if features rarely
co-occur, the network can afford to let their representations overlap.

In [2]:
n_features = 8
n_hidden = 4
p_active = 0.05

dist = SparseUniform(n_features=n_features, p_active=p_active, device="cpu")

# Draw a small batch to see what the data looks like
samples = dist.sample(batch_size=5)
print("Sample shape:", samples.shape)
print("Expected active features per sample (L0):", dist.expected_l0)
print()
print("Five samples (rows=samples, cols=features):")
print(samples.numpy().round(3))
print()
# Count non-zeros to verify sparsity
nonzeros = (samples > 0).sum(dim=1).float()
print(f"Active features per sample: {nonzeros.tolist()}")

Sample shape: torch.Size([5, 8])
Expected active features per sample (L0): 0.4000000059604645

Five samples (rows=samples, cols=features):
[[0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.202 0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.   ]]

Active features per sample: [0.0, 0.0, 0.0, 1.0, 0.0]


## 2. Create an autoencoder

`TiedLinearRelu` is a single-layer autoencoder with tied weights and a
ReLU activation on the decoder output:

    encode(x) = x @ W.T          (n_features -> n_hidden)
    decode(z) = ReLU(z @ W + b)  (n_hidden -> n_features)

The key constraint: `n_features=8 > n_hidden=4`, so the encoder must
compress 8 features into 4 dimensions.  This is the bottleneck that
forces the network to make representational trade-offs.

In [3]:
ae = TiedLinearRelu(n_features=n_features, n_hidden=n_hidden, device="cpu")

print(f"Autoencoder: {n_features} features -> {n_hidden} hidden dims")
print(f"Compression ratio: {n_features / n_hidden:.1f}x")
print(f"Learnable parameters: {sum(p.numel() for p in ae.parameters())}")

Autoencoder: 8 features -> 4 hidden dims
Compression ratio: 2.0x
Learnable parameters: 40


## 3. Train with ToyModel

`ToyModel` ties together a distribution and an autoencoder.  Calling
`.fit()` runs standard SGD training: sample a batch from the distribution,
encode-decode it, compute MSE loss, and update weights.

In [4]:
model = ToyModel(dist, ae, device="cpu")

losses, _ = model.fit(
    n_epochs=5000,
    batch_size=512,
    learning_rate=1e-3,
)

print(f"Initial loss: {losses[0]:.6f}")
print(f"Final loss:   {losses[-1]:.6f}")

Initial loss: 0.107635
Final loss:   0.005517


### Loss curve

We can plot the training loss to verify convergence.  The loss drops
quickly as the network learns to reconstruct the sparse inputs.

In [5]:
import plotly.io as pio

pio.renderers.default = "png"

import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        y=losses,
        mode="lines",
        line=dict(width=1),
        name="Training loss",
    )
)
fig.update_layout(
    title="Training Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE Loss",
    yaxis_type="log",
    height=400,
    width=600,
)
fig.show()

## 4. Inspect geometric properties

After training, `ToyModel` exposes properties that describe how features
are represented in the hidden space.  These are the core tools for
studying superposition.

### The embedding matrix W

`model.W` is the weight matrix of shape `(n_hidden, n_features)`.
Each column is a feature's embedding vector in the hidden space.

In [6]:
W = model.W
print(f"W shape: {W.shape}  (n_hidden x n_features)")
print()
print("Embedding matrix W:")
print(W.numpy().round(4))

W shape: torch.Size([4, 8])  (n_hidden x n_features)

Embedding matrix W:
[[-0.1952 -0.3287 -0.9075 -0.1868  0.1846  0.9063  0.1928  0.3314]
 [-0.1629 -0.2574 -0.0637  0.9504 -0.9544  0.0648  0.1646  0.2605]
 [-0.9158  0.3962  0.0624 -0.0467  0.0469 -0.0614  0.9179 -0.4013]
 [ 0.3153  0.8151 -0.417   0.2494 -0.2483  0.4182 -0.3159 -0.8243]]


### Feature norms

The norm of each feature's embedding vector tells you how strongly it
is represented.  A norm near 1.0 means the feature is fully represented;
a norm near 0 means the network has effectively dropped it.

With equal importances (the default) and high sparsity, the network
can represent all features with near-unit norm by using antipodal
pairs (see the interference matrix below).

In [7]:
norms = model.feature_norms
print("Feature norms (how strongly each feature is represented):")
for i, n in enumerate(norms.tolist()):
    bar = "#" * int(n * 40)
    print(f"  Feature {i}: {n:.4f}  {bar}")

Feature norms (how strongly each feature is represented):
  Feature 0: 1.0014  ########################################
  Feature 1: 0.9979  #######################################
  Feature 2: 1.0027  ########################################
  Feature 3: 1.0013  ########################################
  Feature 4: 1.0044  ########################################
  Feature 5: 1.0021  ########################################
  Feature 6: 1.0033  ########################################
  Feature 7: 1.0090  ########################################


### Superposition metric

The `superposition` property computes the mean maximum absolute cosine
similarity between feature embeddings.  It ranges from 0 (all features
orthogonal -- no superposition) to 1 (features fully overlapping).

With 8 features in 4 dimensions, the network cannot make all features
orthogonal.  At high sparsity (p_active=0.05), superposition is the
optimal strategy: overlap features that rarely co-occur.

In [8]:
rho = model.superposition.item()
print(f"Superposition (rho_mm): {rho:.4f}")
if rho > 0.5:
    print("  -> High superposition: features are overlapping significantly.")
elif rho > 0.1:
    print("  -> Moderate superposition: some feature overlap.")
else:
    print("  -> Low superposition: features are nearly orthogonal.")

Superposition (rho_mm): 1.0000
  -> High superposition: features are overlapping significantly.


### Feature dimensionalities

Each feature occupies a certain number of "effective dimensions" in
the hidden space.  In a 4D space with 8 features, if all features
are equally represented, each gets about 0.5 dimensions on average.
Highly superposed features share dimensions.

In [9]:
dims = model.feature_dimensionalities
print("Feature dimensionalities (effective dims per feature):")
for i, d in enumerate(dims.tolist()):
    bar = "#" * int(d * 40)
    print(f"  Feature {i}: {d:.4f}  {bar}")
print(f"\nMean: {dims.mean():.4f}")
print(f"Sum:  {dims.sum():.4f}  (hidden dims = {n_hidden})")

Feature dimensionalities (effective dims per feature):
  Feature 0: 0.4990  ###################
  Feature 1: 0.4944  ###################
  Feature 2: 0.5003  ####################
  Feature 3: 0.4984  ###################
  Feature 4: 0.5016  ####################
  Feature 5: 0.4997  ###################
  Feature 6: 0.5009  ####################
  Feature 7: 0.5056  ####################

Mean: 0.5000
Sum:  4.0000  (hidden dims = 4)


### Interference matrix

The `interferences` matrix shows how much each pair of features
interferes with each other.  Diagonal entries are self-similarity
(always 1.0 for normalized features); off-diagonal entries reveal
which features the network has allowed to overlap.

In this case you should see **antipodal pairs**: pairs of features
with cosine similarity near -1.0 (e.g. features 0 & 6, 2 & 5, etc.).
The network packs 8 features into 4 dimensions by placing each pair
at opposite ends of the same axis.  The ReLU in the decoder ensures
that only the correct feature is reconstructed for a given input.

In [10]:
interference = model.interferences
print("Interference matrix (cosine similarities between features):")
print(interference.numpy().round(3))

Interference matrix (cosine similarities between features):
[[ 1.001e+00  0.000e+00 -1.000e-03  3.000e-03 -2.000e-03  1.000e-03
  -1.003e+00  1.000e-03]
 [ 0.000e+00  9.980e-01 -1.000e-03  1.000e-03  1.000e-03  2.000e-03
   0.000e+00 -1.009e+00]
 [-1.000e-03 -1.000e-03  1.003e+00  2.000e-03 -0.000e+00 -1.002e+00
   4.000e-03  1.000e-03]
 [ 3.000e-03  1.000e-03  2.000e-03  1.001e+00 -1.004e+00 -1.000e-03
  -1.000e-03 -1.000e-03]
 [-2.000e-03  1.000e-03 -0.000e+00 -1.001e+00  1.004e+00 -1.000e-03
  -0.000e+00 -2.000e-03]
 [ 1.000e-03  2.000e-03 -1.003e+00 -1.000e-03 -1.000e-03  1.002e+00
  -3.000e-03 -3.000e-03]
 [-1.001e+00  0.000e+00  4.000e-03 -1.000e-03 -0.000e+00 -3.000e-03
   1.003e+00 -1.000e-03]
 [ 1.000e-03 -9.980e-01  1.000e-03 -1.000e-03 -2.000e-03 -3.000e-03
  -1.000e-03  1.009e+00]]


## 5. Visualize embeddings in 2D

To directly see where features land in the hidden space, we train a
model with `n_hidden=2`.  Each feature's embedding is a 2D vector
that we can draw as an arrow from the origin.

`plot_embedding` creates an interactive plotly figure with one arrow
per feature.  With high sparsity, you should see features spreading
out in many directions -- this is superposition in action.

In [11]:
# Train a 2D model for visualization
dist_2d = SparseUniform(n_features=n_features, p_active=p_active, device="cpu")
ae_2d = TiedLinearRelu(n_features=n_features, n_hidden=2, device="cpu")
model_2d = ToyModel(dist_2d, ae_2d, device="cpu")

model_2d.fit(n_epochs=5000, batch_size=512, learning_rate=1e-3)

print(f"2D model superposition: {model_2d.superposition.item():.4f}")
print(f"2D model final loss: unneeded (see plot)")

fig = plot_embedding(model_2d)
fig.update_layout(
    title="Feature Embeddings in 2D Hidden Space (high sparsity)",
    height=500,
    width=500,
)
fig.show()

2D model superposition: 0.9577
2D model final loss: unneeded (see plot)


## 6. Explore the sparsity-superposition trade-off

The key insight from the Toy Models of Superposition paper: sparsity
controls how much superposition a network uses.

- **High sparsity** (p_active=0.01): features rarely co-occur, so the
  network freely overlaps them.  More features fit in fewer dims.
- **Low sparsity** (p_active=0.5): features often co-occur, so overlap
  causes large reconstruction errors.  The network may shrink feature
  norms (effectively dropping some features) rather than finding
  orthogonal arrangements, especially when `n_features >> n_hidden`.

Let's train both and compare.

In [12]:
results = {}

for label, p in [("High sparsity (p=0.01)", 0.01), ("Low sparsity (p=0.5)", 0.5)]:
    d = SparseUniform(n_features=n_features, p_active=p, device="cpu")
    a = TiedLinearRelu(n_features=n_features, n_hidden=2, device="cpu")
    m = ToyModel(d, a, device="cpu")
    losses_i, _ = m.fit(n_epochs=5000, batch_size=512, learning_rate=1e-3)

    results[label] = {
        "model": m,
        "final_loss": losses_i[-1],
        "superposition": m.superposition.item(),
        "mean_norm": m.feature_norms.mean().item(),
        "mean_dimensionality": m.feature_dimensionalities.mean().item(),
    }

In [13]:
# -- Compare metrics side by side --
print(f"{'Metric':<30} {'High sparsity':>15} {'Low sparsity':>15}")
print("-" * 62)
for metric in ["final_loss", "superposition", "mean_norm", "mean_dimensionality"]:
    high = results["High sparsity (p=0.01)"][metric]
    low = results["Low sparsity (p=0.5)"][metric]
    print(f"{metric:<30} {high:>15.4f} {low:>15.4f}")

Metric                           High sparsity    Low sparsity
--------------------------------------------------------------
final_loss                              0.0095          0.5899
superposition                           0.9997          0.9990
mean_norm                               1.0395          0.5391
mean_dimensionality                     0.2498          0.2499


Both models show very high superposition (~1.0) because `n_features`
far exceeds `n_hidden` in both cases.  The real difference is in
**mean_norm**: the high-sparsity model keeps all features at full
strength (~1.0) while the low-sparsity model shrinks norms (~0.5),
effectively under-representing features it cannot reconstruct well.
The low-sparsity model also has much higher final loss, reflecting
the unavoidable interference when features frequently co-occur.

In [14]:
# -- Visualize both embeddings --
fig_high = plot_embedding(results["High sparsity (p=0.01)"]["model"])
fig_high.update_layout(
    title="High Sparsity (p=0.01) -- features spread via superposition",
    height=450,
    width=450,
)
fig_high.show()

fig_low = plot_embedding(results["Low sparsity (p=0.5)"]["model"])
fig_low.update_layout(
    title="Low Sparsity (p=0.5) -- features shrink in norm",
    height=450,
    width=450,
)
fig_low.show()

## Summary

You have seen the core occhio workflow:

1. **Define** a sparse data distribution with `SparseUniform`
2. **Build** an autoencoder bottleneck with `TiedLinearRelu`
3. **Train** by combining them in a `ToyModel` and calling `.fit()`
4. **Analyze** the learned geometry with properties like
   `superposition`, `feature_norms`, `feature_dimensionalities`,
   and `interferences`
5. **Visualize** feature embeddings with `plot_embedding`
6. **Compare** how sparsity controls the degree of superposition

Next steps:
- Try different autoencoder architectures (`TiedLinear`, `TiedMLPEncoder`)
- Vary feature importances with the `importances` parameter
- Use `ModelGrid` to sweep over parameters systematically
- Explore correlated features with `CorrelatedPairs` or
  `HierarchicalPairs` distributions